# Sketch2Shape AI - consolidated TripoSR GPU workflow
Run cells in order in a Colab GPU runtime. This consolidates the user-verified experiment: Python **3.12.13**, PyTorch CUDA **12.8**, Transformers **4.40.2**, Tokenizers **0.19.1**, Hugging Face Hub **0.25.2**, compiled torchmcubes, mesh resolution **128**, chunk size **1024**. The exact torch version was not recorded; this notebook keeps and prints the installed working torch/CUDA.

Background removal defaults to **enabled**. The successful diagnostic disabled removal, but its white-background chair sketch had poor fidelity. Change `REMOVE_BACKGROUND` in the settings cell to compare; neither mode guarantees fidelity. No weights are trained or edited. No paid API, server, tunnel or Gradio UI. The downloaded experimental notebook is preserved; use this canonical notebook for execution.

In [ ]:
import os, sys, subprocess, platform, shutil, re
from pathlib import Path

def run_checked(command, **kwargs):
    try:
        return subprocess.run(command, check=True, **kwargs)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError) as exc:
        raise RuntimeError(f"Setup failed: {command[0]}. Read the command output above; stop here and fix the environment before continuing. {exc}") from exc

import torch
print("Python:", sys.version, "Platform:", platform.platform())
print("PyTorch:", torch.__version__, "PyTorch CUDA:", torch.version.cuda)
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("This dependency recipe targets Python 3.10–3.12. Select a compatible Colab runtime before continuing.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU unavailable. Select a GPU runtime and rerun this cell.")
if sys.version_info[:3] != (3, 12, 13) or torch.version.cuda != "12.8":
    print("Verified experiment used Python 3.12.13 / PyTorch CUDA 12.8; this environment differs. No torch/CUDA replacement will be attempted.")
print("GPU:", torch.cuda.get_device_name(0))
run_checked(["nvidia-smi"])
if not shutil.which("nvcc"):
    raise RuntimeError("CUDA toolkit/nvcc is required to compile torchmcubes.")
nvcc = subprocess.check_output(["nvcc", "--version"], text=True)
print(nvcc)
match = re.search(r"release (\d+)\.", nvcc)
if not match or not torch.version.cuda or match.group(1) != torch.version.cuda.split('.')[0]:
    raise RuntimeError("CUDA toolkit and PyTorch CUDA major versions must match. Use a compatible runtime; do not blindly replace torch.")

## Obtain your project source — choose ZIP or Drive (no credentials in code)

In [ ]:
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

def extract_project_zip(archive_file, destination):
    """Normalize Windows ZIP separators, reject unsafe paths, never overwrite."""
    root = Path(destination).resolve()
    if root.exists():
        raise FileExistsError(f"Extraction target already exists: {root}")
    with zipfile.ZipFile(archive_file) as archive:
        entries = archive.infolist()
        if sum(entry.file_size for entry in entries) > 512 * 1024**2:
            raise ValueError("ZIP exceeds 512 MiB unpacked; omit environments/weights/outputs")
        normalized = []
        seen = set()
        for entry in entries:
            name = entry.filename.replace("\\", "/")
            parts = PurePosixPath(name).parts
            if (not parts or name.startswith("/") or ".." in parts
                    or any(":" in part for part in parts)
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {entry.filename!r}")
            target = root.joinpath(*parts)
            if not target.resolve().is_relative_to(root):
                raise ValueError("ZIP path escapes extraction directory")
            if target in seen:
                raise ValueError("ZIP contains duplicate normalized paths")
            seen.add(target)
            normalized.append((entry, target, name.endswith("/")))
        root.mkdir(parents=True)
        for entry, target, is_directory in normalized:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(entry) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
    matches = list(root.rglob("backend/generator.py"))
    if len(matches) != 1:
        raise ValueError(f"Expected one project backend/generator.py, found {len(matches)}")
    return matches[0].parent.parent.resolve()

In [ ]:
SOURCE_MODE = "zip"  # or "drive"
DRIVE_PROJECT = "/content/drive/MyDrive/Sketch2Shape"
if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path(DRIVE_PROJECT).resolve()
elif SOURCE_MODE == "zip":
    from google.colab import files
    import io, tempfile
    print("Upload one CURRENT project ZIP including backend/requirements-colab.txt; omit environments, secrets and outputs.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one ZIP")
    source_root = Path(tempfile.mkdtemp(prefix="sketch2shape-", dir="/content")) / "source"
    PROJECT = extract_project_zip(io.BytesIO(next(iter(uploaded.values()))), source_root)
    del uploaded
else:
    raise ValueError("SOURCE_MODE must be zip or drive")
if not (PROJECT / "backend/generator.py").is_file() or not (PROJECT / "backend/requirements-colab.txt").is_file():
    raise FileNotFoundError("Upload the UPDATED backend, including requirements-colab.txt")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)

## Install the consolidated dependency recipe
Keep Colab's installed CUDA-enabled torch. The three Hugging Face packages use the exact working experiment versions. Build tools include **scikit-build-core and pybind11**, missing in the failed initial setup. torchmcubes is compiled against this runtime's torch/CUDA and tested with a small CUDA marching-cubes operation (no model download). Failures stop with console diagnostics. The local test environment stays lightweight.

In [ ]:
TRIPOSR = Path("/content/TripoSR")
if not TRIPOSR.exists():
    run_checked(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", str(TRIPOSR)], timeout=300)
else:
    origin = subprocess.check_output(["git", "-C", str(TRIPOSR), "remote", "get-url", "origin"], text=True).strip()
    if origin != "https://github.com/VAST-AI-Research/TripoSR.git":
        raise RuntimeError("Existing /content/TripoSR has an unexpected origin")
UPSTREAM_COMMIT = subprocess.check_output(["git", "-C", str(TRIPOSR), "rev-parse", "HEAD"], text=True).strip()
print("Upstream commit:", UPSTREAM_COMMIT)
import tempfile
# Prevent pip from replacing the working torch through a transitive dependency.
constraints = Path(tempfile.gettempdir()) / "sketch2shape-torch-constraint.txt"
constraints.write_text(f"torch=={torch.__version__}\n")
run_checked([sys.executable, "-m", "pip", "install", "setuptools", "wheel", "ninja", "cmake", "packaging", "scikit-build-core", "pybind11"], timeout=600)
run_checked([sys.executable, "-m", "pip", "install", "-c", str(constraints),
             "-r", str(PROJECT / "backend/requirements-colab.txt")], timeout=900)
os.environ["CUDA_HOME"] = str(Path(shutil.which("nvcc")).resolve().parents[1])
os.environ["TORCH_CUDA_ARCH_LIST"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
os.environ["MAX_JOBS"] = "2"
# Rebuild only this native extension for the runtime's torch/CUDA architecture.
run_checked([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "--no-cache-dir", "--force-reinstall",
             "git+https://github.com/tatsy/torchmcubes.git"], timeout=1200)
os.environ["TRIPOSR_DIR"] = str(TRIPOSR)
check_code = """
import torch, numpy, transformers, tokenizers, huggingface_hub, rembg, xatlas, moderngl
from tsr.system import TSR
from torchmcubes import marching_cubes
import torchmcubes_module as m
assert torch.cuda.is_available()
assert hasattr(m, 'mcubes_cuda'), 'torchmcubes lacks CUDA support'
assert transformers.__version__ == '4.40.2'
assert tokenizers.__version__ == '0.19.1'
assert huggingface_hub.__version__ == '0.25.2'
field = torch.zeros((8, 8, 8), device='cuda')
field[2:6, 2:6, 2:6] = 1
vertices, faces = marching_cubes(field, 0.5)
assert len(vertices) and len(faces), 'CUDA marching cubes smoke check failed'
print('Dependency/CUDA checks passed:', torch.__version__, torch.version.cuda)
print('HF versions:', transformers.__version__, tokenizers.__version__, huggingface_hub.__version__)
"""
run_checked([sys.executable, "-c", check_code], cwd=str(TRIPOSR), timeout=180)
run_checked([sys.executable, str(TRIPOSR / "run.py"), "--help"], cwd=str(TRIPOSR), timeout=120)
print("TRIPOSR_DIR:", os.environ["TRIPOSR_DIR"])
print("Inference/checks use fresh subprocesses to avoid stale kernel imports after pip installation.")

## Offline backend checks — no model weights or inference

In [ ]:
run_checked([sys.executable, "-m", "compileall", "-q", "backend"], cwd=str(PROJECT), timeout=60)
run_checked([sys.executable, "-m", "unittest", "discover", "-s", "backend/tests", "-v"], cwd=str(PROJECT), timeout=120)
# Running checks in subprocesses avoids reusing stale NumPy/Pillow modules after pip installation.

## Choose a fresh input image (or the included sketch)

In [ ]:
USE_UPLOADED_IMAGE = True
if USE_UPLOADED_IMAGE:
    from google.colab import files
    uploaded_images = files.upload()
    if len(uploaded_images) != 1:
        raise ValueError("Upload exactly one PNG/JPEG/WebP image.")
    import uuid
    input_dir = PROJECT / "examples"
    input_dir.mkdir(exist_ok=True)
    name, data = next(iter(uploaded_images.items()))
    IMAGE = input_dir / f"uploaded_{uuid.uuid4().hex}{Path(name).suffix.lower()}"
    IMAGE.write_bytes(data)
    del uploaded_images
else:
    IMAGE = PROJECT / "examples/chair.png"
if not IMAGE.is_file():
    raise FileNotFoundError(IMAGE)
print("Input:", IMAGE)
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(IMAGE)))

## Inference settings
Background removal is configurable, not forced off. Higher resolution can increase RAM/VRAM sharply and does not resolve sparse-sketch ambiguity.

In [ ]:
# Conservative settings from the successful low-memory diagnostic.
MC_RESOLUTION = 128
CHUNK_SIZE = 1024
REMOVE_BACKGROUND = True  # False only for deliberately prepared RGB/gray-background inputs.
TIMEOUT_SECONDS = 1800
os.environ.update({
    "TRIPOSR_MC_RESOLUTION": str(MC_RESOLUTION),
    "TRIPOSR_CHUNK_SIZE": str(CHUNK_SIZE),
    "TRIPOSR_REMOVE_BG": str(REMOVE_BACKGROUND).lower(),
    "TRIPOSR_TIMEOUT_SECONDS": str(TIMEOUT_SECONDS),
})
print({key: value for key, value in os.environ.items() if key.startswith("TRIPOSR_")})

## Explicit real inference
First invocation downloads public pretrained weights through upstream run.py (and background-removal weights when enabled). `generate_3d()` creates the required `0/` export folder, validates the real GLB and only then returns its absolute path. No static mesh or alternate engine is substituted. This revision consolidates the successful diagnostic but has only been tested offline locally.

In [ ]:
import time, json, uuid
result_record = PROJECT / "outputs" / f"notebook_result_{uuid.uuid4().hex}.json"
GLB = None  # Failed reruns must not offer an old result for download.
started = time.perf_counter()
inference_code = """
import json, logging, sys, time
from pathlib import Path
from backend.generator import generate_3d
from backend.validate_mesh import validate_mesh
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
start = time.perf_counter()
path = generate_3d(sys.argv[1], prompt=None)
info = validate_mesh(path)
assert Path(path).is_file() and Path(path).stat().st_size > 0
Path(sys.argv[2]).write_text(json.dumps({'path': path, 'mesh': info, 'seconds': time.perf_counter() - start}))
"""
run_checked([sys.executable, "-c", inference_code, str(IMAGE), str(result_record)], cwd=str(PROJECT), timeout=int(os.environ["TRIPOSR_TIMEOUT_SECONDS"]) + 60)
result = json.loads(result_record.read_text())
GLB = Path(result["path"])
if not GLB.is_file() or GLB.stat().st_size == 0:
    raise RuntimeError("Validated GLB is no longer present")
print("Validated GLB:", GLB)
print("Mesh information:", result["mesh"])
print("Inference + validation seconds:", round(result["seconds"], 2))
print("Total cell seconds:", round(time.perf_counter() - started, 2))
print("Bytes:", GLB.stat().st_size)

## Download the validated GLB — click explicitly; save before the runtime resets

In [ ]:
import ipywidgets as widgets
from google.colab import files
button = widgets.Button(description="Download validated GLB", layout=widgets.Layout(width="240px"))
status = widgets.Output()
def download_clicked(_):
    with status:
        if GLB is None or not GLB.is_file():
            raise FileNotFoundError("Run inference successfully before downloading")
        # Revalidate the file immediately before offering the download.
        run_checked([sys.executable, "-c", "import sys; from backend.validate_mesh import validate_mesh; print(validate_mesh(sys.argv[1]))", str(GLB)], cwd=str(PROJECT), timeout=120)
        files.download(str(GLB))
button.on_click(download_clicked)
display(button, status)

### Attribution and limits
[Official TripoSR](https://github.com/VAST-AI-Research/TripoSR) by Tripo AI / Stability AI; [public pretrained weights](https://huggingface.co/stabilityai/TripoSR), MIT license. Single-image reconstruction can be ambiguous, especially for sparse sketches. Geometry validation does not prove resemblance. Upstream defaults need about 6 GB VRAM; free Colab GPUs and session lifetime are limited. This is manual notebook generation and file download: the local frontend cannot directly access a Colab filesystem or execute here. Frontend transport remains future work. No public endpoint is created.